# High Fidelity Data Ingestion for Oracle 23ai

**Copyright 2026, Denis Rothman**

This notebook serves as a production-ready template for Oracle DBAs and AI Engineers to implement **High Fidelity Data Ingestion** pipelines for Retrieval-Augmented Generation (RAG).

It moves beyond simple text storage by implementing a dual-stream ingestion process that handles both **Knowledge** (raw evidence) and **Context** (behavioral blueprints). The workflow demonstrates how to transform unstructured data into high-dimensional vectors and store them securely in Oracle Database 23ai.

### Notebook Highlights:

* **Use Case:** A **"Cyber-Incident Command Center"** scenario. We simulate a complete security breach dataset—containing server logs, Slack transcripts, and phishing reports—to demonstrate how AI can correlate disparate data sources.
* **Infrastructure:** Establishes secure connectivity using Oracle Wallets and the `oracledb` thin client.
* **Data Pipeline:**
* **Staging:** Dynamic generation of 7 distinct "Evidence" files and 3 "Semantic Blueprints."
* **Transformation:** Robust text chunking (via `tiktoken`) and vector embedding (via OpenAI `text-embedding-3-small`).
* **Loading:** Efficient batch insertion into Oracle Vector tables (`KNOWLEDGE_STORE` and `CONTEXT_LIBRARY`).


* **Validation:** Includes a test to perform semantic similarity searches (e.g., *"What happened to the admin database?"*) immediately after ingestion to verify index health.




# 1.Installation and Setup

## DbA Parameters


## 1.1 Oracle environment Setup & Wallet Management

This step prepares the runtime environment by connecting to Google Drive and ensuring the Oracle Wallet is available. The Oracle Wallet contains the SSL/TLS certificates and configuration files (tnsnames.ora, sqlnet.ora) necessary for a secure connection to the Oracle Autonomous Database.

Google Drive Mount: Maps your personal Drive to /content/drive, allowing the notebook to read credentials and configuration files stored externally.

Wallet Unzipping: If unzip_wallet is set to True, the script searches for the wallet ZIP file in the specified path and extracts it. This ensures the connection drivers can locate the required security certificates.

Path Definition: Sets wallet_path to the directory where the unzipped configuration files reside, which will be passed to the Oracle driver in subsequent steps.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
wallet_path = "/content/drive/MyDrive/oracle_wallet"

Mounted at /content/drive


## 1.2 Install Oracle Database Driver
This command installs the oracledb Python driver, which is the renamed and modernized successor to the legacy cx_Oracle driver. It serves as the bridge between this Python notebook and the Oracle Database.

Version Pinning (==3.4.1): The version is explicitly fixed to 3.4.1 to ensure stability and reproducibility. In production DBA scripts, pinning versions prevents unexpected updates or API changes from breaking the automation workflow.

Thin Client Mode: By default, this driver operates in "Thin" mode, meaning it communicates directly with the database using pure Python code without requiring local Oracle Client libraries (Instant Client).

In [ ]:
!pip install oracledb==3.4.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.3 MB/s eta 0:00:00


In [ ]:
import oracledb
print(oracledb.__version__)

3.4.1


### Establish Secure Database Connection
This step handles the authentication and connection to the Oracle 23ai instance. It is designed to adhere to security best practices by strictly separating code from credentials.

External Credential Management: Instead of hardcoding sensitive passwords directly into the notebook (which is a security risk), the script reads a credentials.txt file stored securely on Google Drive.

Credential Parsing: The read_key_value_file helper function parses the external file to retrieve the username, password, Wallet password, and DSN (Data Source Name).

Connection Initialization: The oracledb.connect() method establishes the session using the extracted credentials and the local Wallet configuration path.

Connectivity Test: A simple "Heartbeat" query (SELECT ... FROM DUAL) is executed immediately to verify that the connection is active and stable before proceeding.

In [ ]:
import os

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", "mqyv1gnzq40yxs41_high")
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

('Connected!',)


## 1.3.Installing OpenAI

In [ ]:
!pip install tqdm==4.67.1 --upgrade
!pip install openai==2.14.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.12.0
    Uninstalling openai-2.12.0:
      Successfully uninstalled openai-2.12.0


In [ ]:
# Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")

# Configuration
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"

OpenAI API key loaded and environment variable set successfully.


## 1.4. Additional imports

In [ ]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken                                                         # tokenization
from tenacity import retry, stop_after_attempt, wait_random_exponential # to retry functions
import re
import textwrap
from IPython.display import display, Markdown
import copy

# 3.Data Preparation: The Context Library (Procedural RAG)

### 3.1 Initialize Staging Directory
This step creates a local directory to act as a staging area for our raw data. In a real-world scenario, this might represent a secure S3 bucket or a network drive where logs and reports are aggregated during an incident. We will store our seven "evidence" files here before processing them into the Oracle database.

In [ ]:
# Create a directory to store our source documents (Evidence)
import os

if not os.path.exists("incident_documents"):
    os.makedirs("incident_documents")
    print("Directory 'incident_documents' created.")
else:
    print("Directory 'incident_documents' already exists.")

Directory 'incident_documents' created.


## 3.2.Knowledge Store(the evidence)

In this section, we generate the **evidence** files. These represent the raw, unstructured data that an organization possesses but often struggles to correlate. We will create seven distinct files ranging from technical logs to human conversations.

In [ ]:
#@title Document 1: Server System Logs
'''
This document contains timestamped server logs detailing a security incident that begins with a user login, escalates through SQL syntax errors and performance latency, and culminates in a confirmed SQL injection attack and subsequent service restart
'''
server_syslog_text = """
2025-10-24 03:14:20 [INFO]  AuthService: User authentication successful for user_id=402. IP: 192.168.1.45
2025-10-24 03:15:05 [WARN]  DB_Connector: High latency detected on query_id=9921. Duration: 4500ms.
2025-10-24 03:15:07 [ERROR] AppServer: SQLSyntaxErrorException: ORA-00933: SQL command not properly ended.
2025-10-24 03:15:07 [ERROR] SecurityFilter: Suspicious payload detected in params: "UNION SELECT username, password FROM admin_users --"
2025-10-24 03:15:08 [CRITICAL] DB_Connector: Connection pool exhaustion. Active connections: 500/500.
2025-10-24 03:15:10 [ALERT] IDS_System: Signature match [SQL_INJECTION_GENERIC] from IP: 203.0.113.88.
2025-10-24 03:16:00 [INFO]  SysAdmin: Service restart initiated by watchdog.
"""

with open("incident_documents/Server_SysLog_001.txt", "w") as f:
    f.write(server_syslog_text)

print("✅ Created incident_documents/Server_SysLog_001.txt")

✅ Created incident_documents/Server_SysLog_001.txt


In [ ]:
#@title Document 2: Ops Slack Channel Transcript
'''
This file captures the human element of the incident—the confusion and rapid troubleshooting between engineers as the attack unfolds.
'''

slack_ops_text = """
Channel: #ops-critical
Date: 2025-10-24

[03:14 AM] @dev_sarah: Hey, are we seeing lag on the main DB? My dashboard just lit up red.
[03:15 AM] @dba_mike: Yeah, seeing a massive spike in connections. Investigating.
[03:15 AM] @dev_sarah: I'm getting 500 errors on the login page.
[03:16 AM] @dba_mike: Whoa. I'm seeing a weird query pattern. Someone is hammering the 'admin_users' table.
[03:17 AM] @sec_lead_alex: @dba_mike Post the query signature. Now.
[03:18 AM] @dba_mike: It's a UNION SELECT. Looks like a classic injection attempt. IP 203.0.113.88.
[03:19 AM] @sec_lead_alex: CONFIRMED. That IP is external. Block it at the firewall immediately!
[03:20 AM] @dev_sarah: Firewall rule updated. Traffic dropping off.
[03:22 AM] @dba_mike: Did they get anything?
[03:25 AM] @sec_lead_alex: Too early to tell. We need to dump the access logs and check the exfiltration size. Everyone join the War Room voice channel.
"""

with open("incident_documents/Slack_Channel_Ops.txt", "w") as f:
    f.write(slack_ops_text)

print("✅ Created incident_documents/Slack_Channel_Ops.txt")

✅ Created incident_documents/Slack_Channel_Ops.txt


In [ ]:
#@title Document 3: Security Phishing Analysis Report
'''
This report identifies the root cause of the breach: a successful phishing attack that compromised the credentials used in the logs from Document 1.
'''
phishing_report_text = """
SECURITY INCIDENT REPORT: PHISHING VECTOR ANALYSIS
ID: PHISH-2025-882
Date: 2025-10-23
Severity: HIGH

Overview:
The Security Operations Center (SOC) detected a targeted phishing campaign affecting the Finance and Admin departments.

Email Metadata:
- Subject: "URGENT: Payroll Portal Authentication Required"
- Sender: support@payro11-admin.com (Typosquatting detected: 'l' replaced with '1')
- Recipient: j.smith@company.com (User ID: 402)
- Timestamp Received: 2025-10-23 14:45:00

Payload Analysis:
The email contained a link to a credential harvesting site hosting a fake SSO login page.
URL: http://secure-login-portal-882.xz/auth

Incident Correlation:
User 402 clicked the link at 14:48:00.
SOC observed a successful login from an unrecognized device (IP 203.0.113.88) approximately 12 hours later, coinciding with the SQL injection attempts recorded in System Logs.

Status:
- Malicious domain blocked.
- User 402 account locked and forced password reset.
- Remedial training assigned to affected user.
"""

with open("incident_documents/Email_Phishing_Report.txt", "w") as f:
    f.write(phishing_report_text)

print("✅ Created incident_documents/Email_Phishing_Report.txt")

✅ Created incident_documents/Email_Phishing_Report.txt


In [ ]:
#@title Document 4: Software Patch Release Notes
'''
This technical document connects the "what" (the attack) to the "how" (the fix), confirming that the vulnerability exploited in the logs has been identified and resolved by the engineering team.
'''

patch_notes_text = """
Software Release Notes: Hotfix v2.4.1
Release Date: 2025-10-24
Status: DEPLOYED TO STAGING

Summary:
This emergency hotfix addresses a critical security vulnerability in the user authentication module.

Changelog:
- Security Fix (CVE-2025-9921): Patched a SQL Injection vulnerability in `AdminLoginController.java`.
- Update Detail: Implemented prepared statements for all database queries involving user input. Removed legacy string concatenation logic in the login query builder.
- Performance: Added connection pool throttling to prevent resource exhaustion during high-load events (or DoS attempts).

Verification:
- Penetration test suite `Sim_Attack_SQLi` passed.
- Regression testing on User Login module passed.

Deployment Schedule:
- Staging: Complete.
- Production: Scheduled for 04:00 AM maintenance window (pending final approval).
"""

with open("incident_documents/Patch_Release_Notes.txt", "w") as f:
    f.write(patch_notes_text)

print("✅ Created incident_documents/Patch_Release_Notes.txt")

✅ Created incident_documents/Patch_Release_Notes.txt


In [ ]:
#@title Document 5: Customer Support Ticket Dump
'''
This file represents the "downstream impact" of the incident. It captures the confusion and frustration of users experiencing the service outage caused by the database lockup.
'''

support_tickets_text = """
Ticket_ID,Timestamp,Customer_ID,Subject,Message
TKT-9001,2025-10-24 03:16:00,Cust_554,URGENT: 500 Error,I cannot log in to the admin panel. Getting a blank screen with 'Internal Server Error'.
TKT-9002,2025-10-24 03:20:00,Cust_112,Is the system down?,My dashboard isn't loading data. I have payroll to run!
TKT-9003,2025-10-24 03:25:00,Cust_899,Suspicious Email?,I just heard from a colleague about a weird email asking for login info. Did we get hacked?
TKT-9004,2025-10-24 03:45:00,Cust_554,Still broken,Any ETA on the fix? This downtime is costing us money.
TKT-9005,2025-10-24 04:05:00,Cust_300,Login Works But Slow,I managed to log in finally, but everything is crawling. Is it safe to use?
"""

with open("incident_documents/Customer_Support_Tickets.txt", "w") as f:
    f.write(support_tickets_text)

print("✅ Created incident_documents/Customer_Support_Tickets.txt")

✅ Created incident_documents/Customer_Support_Tickets.txt


In [ ]:
#@title Document 6: Internal Data Privacy & Compliance Policy
'''
This is the "rulebook" the AI will need to consult to determine if the incident described in the logs and tickets actually constitutes a legal breach that must be reported.
'''
privacy_policy_text = """
INTERNAL POLICY DOCUMENT: DATA PRIVACY & INCIDENT REPORTING
Policy ID: LEG-COMP-005
Effective Date: 2024-01-01

1. Definition of Personal Data Breach (GDPR/CCPA):
A breach is defined as any security incident that leads to the accidental or unlawful destruction, loss, alteration, unauthorized disclosure of, or access to, personal data transmitted, stored, or otherwise processed.

2. Reporting Timelines:
- Internal: All potential breaches must be reported to the Data Protection Officer (DPO) within 1 hour of detection.
- External (Regulatory): In the event of a confirmed breach involving unencrypted personal data (names, emails, IDs), the organization must notify the relevant supervisory authority without undue delay and, where feasible, not later than 72 hours after having become aware of it.

3. Classification of Data Sensitivity:
- Tier 1 (Public): Marketing materials, press releases. (No reporting required).
- Tier 2 (Internal): Slack logs, internal memos. (Internal reporting only).
- Tier 3 (Confidential): Customer names, email addresses, support tickets. (Requires Regulatory Reporting if leaked).
- Tier 4 (Restricted): Passwords, Financial Data, Biometrics. (Requires Immediate Regulatory & Customer Notification).

4. Incident Response Protocol:
In the event of a Tier 3 or Tier 4 breach, Legal Counsel must approve all external communications to ensure liability is minimized while maintaining compliance.
"""

with open("incident_documents/Internal_Policy_DataPrivacy.txt", "w") as f:
    f.write(privacy_policy_text)

print("✅ Created incident_documents/Internal_Policy_DataPrivacy.txt")

✅ Created incident_documents/Internal_Policy_DataPrivacy.txt


In [ ]:
#@title Document 7: The Hacker's Manifesto (Artifact)
'''
This adds a narrative layer to the dataset—a taunting note left by the attacker, confirming the motive (ransom) and the specific target (the admin database), which correlates with the logs and the privacy policy risks.
'''
manifesto_text = """
To the "Admins" of this secure facility:

You really should sanitize your inputs.
Did you think a simple firewall would stop us?
Your 'admin_users' table was wide open. We walked right in.

We have your customer list. We have the hashes.
We are generous, though. We will give you 24 hours.
Send 50 BTC to the wallet address below, or we release the full dump to the public.

Tick tock.

- The NullPointer Crew
"""

with open("incident_documents/Hacker_Manifesto.txt", "w") as f:
    f.write(manifesto_text)

print("✅ Created incident_documents/Hacker_Manifesto.txt")

✅ Created incident_documents/Hacker_Manifesto.txt


## 3.2.The Context Library(the instructions)

This section defines the Semantic Blueprints. While the Knowledge Store contains the facts (logs, reports), these blueprints contain the behavior.

We are creating three distinct "personas" for the AI:

Public Relations Manager: Generates reassuring, non-technical updates for customers.

Lead Engineer: Generates detailed, technical Root Cause Analysis (RCA) reports.

Legal Counsel: Generates formal risk assessments based on compliance policies.

In [ ]:
# We define the Semantic Blueprints derived from our Cyber-Incident use case.
# The 'description' allows the Librarian agent to select the right format
# (e.g., "Draft a PR statement" vs "Analyze the root cause").

context_blueprints = [
    {
        "id": "blueprint_public_pr",
        "description": "A blueprint for drafting public-facing statements or press releases regarding service outages or security incidents. The goal is to reassure customers without admitting specific technical liabilities until confirmed.",
        "blueprint": json.dumps({
              "scene_goal": "Manage public perception and reassure customers.",
              "style_guide": "Use a professional, empathetic, but vague tone. Avoid technical jargon like 'SQL Injection' or 'Hacks'. Focus on 'Security Maintenance' or 'Unplanned Outage'.",
              "structure": ["Acknowledgment of Issue", "Current Status (Investigating)", "Estimated Resolution Time", "Apology"],
              "instruction": "Draft a public statement based on the provided facts. Do not reveal sensitive internal details (IPs, specific vulnerabilities)."
            })
    },
    {
        "id": "blueprint_technical_rca",
        "description": "A blueprint for creating a technical Root Cause Analysis (RCA) for engineering and security teams. Focuses on specific timestamps, error codes, vulnerabilities (CVEs), and remediation steps.",
        "blueprint": json.dumps({
              "scene_goal": "Provide a precise technical post-mortem of the incident.",
              "style_guide": "Clinical, precise, and fact-based. Use active voice. Explicitly reference log timestamps, error codes (ORA-XXXX), and IP addresses.",
              "structure": ["Incident Timeline", "Root Cause (The Vulnerability)", "Attack Vector", "Remediation (The Fix)"],
              "instruction": "Synthesize the provided logs and chat transcripts into a formal RCA document."
            })
    },
    {
        "id": "blueprint_legal_compliance",
        "description": "A blueprint for assessing legal liability and data breach notification obligations. It compares incident facts against internal privacy policies (GDPR/CCPA) to determine if a regulatory filing is required.",
        "blueprint": json.dumps({
              "scene_goal": "Determine legal risk and reporting requirements.",
              "style_guide": "Formal, risk-averse, and citation-heavy. Reference specific clauses from the provided Policy Documents.",
              "structure": ["Incident Summary", "Data Impact Assessment (Tier 1-4)", "Policy Violation Check", "Required Actions (Notification Yes/No)"],
              "instruction": "Analyze the incident against the 'Internal Data Privacy & Compliance Policy'. Determine if the compromised data requires external reporting."
            })
    }
]

print(f"\nPrepared {len(context_blueprints)} context blueprints.")


Prepared 3 context blueprints.


## 4.Updating the Data Loading and Processing Logic


This script iterates through the incident_documents directory we created in step 3.1. It reads every .txt file—logs, transcripts, and reports and loads them into a Python dictionary called knowledge_base.

This mimics a real-world ETL (Extract, Transform, Load) process where a DBA might script the ingestion of flat files from a server directory before processing them into the database.

In [ ]:
# 3.4. Load Evidence into Memory
# -------------------------------------------------------------------------
# Load all documents from our new directory
knowledge_base = {}
doc_dir = "incident_documents"  # Updated to match our new use case

if os.path.exists(doc_dir):
    for filename in os.listdir(doc_dir):
        if filename.endswith(".txt"):
            with open(os.path.join(doc_dir, filename), 'r') as f:
                knowledge_base[filename] = f.read()

    print(f"📚 Loaded {len(knowledge_base)} documents into the knowledge base.")
else:
    print(f"❌ Directory '{doc_dir}' not found. Please run the document creation cells above.")

📚 Loaded 7 documents into the knowledge base.


# 5.Helper Functions for Chunking and Embedding

Before we can upload data to Oracle, we need two critical utilities:

chunk_text: A function to break large documents (like the "Patch Notes") into smaller, manageable pieces that fit within the AI's context window. We use tiktoken to ensure we split by tokens, not just characters.

get_embeddings_batch: A function that sends text to OpenAI's API and returns the vector representation (VECTOR(1536)). This is the bridge between human language and the Oracle Vector database.

In [ ]:
# 4. Helper Functions for Chunking and Embedding
# -------------------------------------------------------------------------

# Initialize tokenizer for robust, token-aware chunking
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=400, overlap=50):
    """
    Chunks text based on token count with overlap.
    Best practice for RAG: Overlap ensures context isn't lost at the cut.
    """
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        # Basic cleanup to remove excessive newlines
        chunk_text = chunk_text.replace("\n", " ").strip()
        if chunk_text:
            chunks.append(chunk_text)
    return chunks

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    """
    Generates embeddings for a batch of texts using OpenAI, with exponential backoff retries.
    """
    # OpenAI recommends replacing newlines with spaces for best embedding results
    texts = [t.replace("\n", " ") for t in texts]

    # Call the OpenAI API
    response = client.embeddings.create(input=texts, model=model)

    # Return just the list of embedding vectors
    return [item.embedding for item in response.data]

# 6.Processing and uploading the data to Oracle

In [ ]:
#@title 6.1.Processing and uploading Context Library to Oracle
import oracledb

print("\nProcessing and uploading Context Library to Oracle...")

# 1. Initialize the cursor
cursor = connection.cursor()

# 2. CRITICAL: Tell Oracle that the "embedding" list is actually a VECTOR type
# Without this, the driver thinks the list is a PL/SQL array (causing ORA-01484)
cursor.setinputsizes(embedding=oracledb.DB_TYPE_VECTOR)

vectors_context = []
for item in tqdm(context_blueprints):
    embedding = get_embeddings_batch([item['description']])[0]
    vectors_context.append({
        "id": item['id'],
        "values": embedding,
        "metadata": {
            "description": item['description'],
            "blueprint_json": item['blueprint']
        }
    })

if vectors_context:
    for item in vectors_context:
        cursor.execute("""
            INSERT INTO context_library (id, description, blueprint_json, embedding)
            VALUES (:id, :description, :blueprint_json, :embedding)
        """,
        {
            "id": item["id"],
            "description": item["metadata"]["description"],
            "blueprint_json": item["metadata"]["blueprint_json"],
            "embedding": item["values"]   # Now correctly interpreted as VECTOR
        })

    connection.commit()
    print(f"Successfully uploaded {len(vectors_context)} context vectors to Oracle.")


Processing and uploading Context Library to Oracle...


  0%|          | 0/3 [00:00<?, ?it/s]

Successfully uploaded 3 context vectors to Oracle.


In [ ]:
#@title 6.2.Processing and uploading Knowledge Base to Oracle
import oracledb

print("\nProcessing and uploading Knowledge Base to Oracle...")

# 1. Initialize the cursor explicitly to ensure it exists
cursor = connection.cursor()

# 2. Tell Oracle that the "embedding" bind variable is a VECTOR type
cursor.setinputsizes(embedding=oracledb.DB_TYPE_VECTOR)

batch_size = 100
total_vectors_uploaded = 0

for doc_name, doc_content in knowledge_base.items():
    print(f"  - Processing document: {doc_name}")

    knowledge_chunks = chunk_text(doc_content)

    for i in tqdm(range(0, len(knowledge_chunks), batch_size), desc=f"  Uploading {doc_name}"):
        batch_texts = knowledge_chunks[i:i+batch_size]
        batch_embeddings = get_embeddings_batch(batch_texts)

        for j, embedding in enumerate(batch_embeddings):
            chunk_id = f"{doc_name}_chunk_{total_vectors_uploaded + j}"

            cursor.execute("""
                INSERT INTO knowledge_store (id, source, text, embedding)
                VALUES (:id, :source, :text, :embedding)
            """,
            {
                "id": chunk_id,
                "source": doc_name,
                "text": batch_texts[j],
                "embedding": embedding
            })

        connection.commit()

    total_vectors_uploaded += len(knowledge_chunks)

print(f"\nSuccessfully uploaded {total_vectors_uploaded} knowledge vectors to Oracle.")


Processing and uploading Knowledge Base to Oracle...
  - Processing document: Email_Phishing_Report.txt


  Uploading Email_Phishing_Report.txt:   0%|          | 0/1 [00:00<?, ?it/s]

  - Processing document: Patch_Release_Notes.txt


  Uploading Patch_Release_Notes.txt:   0%|          | 0/1 [00:00<?, ?it/s]

  - Processing document: Slack_Channel_Ops.txt


...

  - Processing document: Internal_Policy_DataPrivacy.txt


  Uploading Internal_Policy_DataPrivacy.txt:   0%|          | 0/1 [00:00<?, ?it/s]

  - Processing document: Server_SysLog_001.txt


  Uploading Server_SysLog_001.txt:   0%|          | 0/1 [00:00<?, ?it/s]


Successfully uploaded 7 knowledge vectors to Oracle.


# 7.Final Verification


This step performs a live test of the vector index. We use a specific question—"What happened to the admin database?"—to verify that the database can semantically link a natural language query to the raw evidence files (like the Server Logs and Slack Transcripts) we just uploaded.

Embedding: The query is converted into a vector using the same model as our data.

VECTOR_DISTANCE: Oracle calculates the distance between our query vector and every stored document vector.

Ranking: The results are ordered by the <-> operator (Euclidean distance), bringing the most relevant "evidence" to the top.

In [ ]:
import oracledb

# Prepare a test query
test_query = "What happened to the admin database?"
query_embedding = get_embeddings_batch([test_query])[0]

cursor.setinputsizes(query_vec=oracledb.DB_TYPE_VECTOR)

cursor.execute("""
    SELECT id, source, text,
           VECTOR_DISTANCE(embedding, :query_vec) AS distance
    FROM knowledge_store
    ORDER BY embedding <-> :query_vec
    FETCH FIRST 5 ROWS ONLY
""",
{
    "query_vec": query_embedding
})

results = cursor.fetchall()

print("\nTop 5 vector search results:\n")
for r in results:
    text_clob = r[2]              # this is a CLOB
    text_str = text_clob.read()   # convert to Python string

    print(f"ID: {r[0]}")
    print(f"Source: {r[1]}")
    print(f"Text snippet: {text_str[:120]}...")
    print(f"Distance: {r[3]}")
    print("-" * 60)


Top 5 vector search results:

ID: Server_SysLog_001.txt_chunk_6
Source: Server_SysLog_001.txt
Text snippet: 2025-10-24 03:14:20 [INFO]  AuthService: User authentication successful for user_id=402. IP: 192.168.1.45 2025-10-24 03:...
Distance: 0.5644098623355615
------------------------------------------------------------
ID: Hacker_Manifesto.txt_chunk_4
Source: Hacker_Manifesto.txt
Text snippet: To the "Admins" of this secure facility:  You really should sanitize your inputs. Did you think a simple firewall would ...
Distance: 0.6007344530077168
------------------------------------------------------------
ID: Customer_Support_Tickets.txt_chunk_3
Source: Customer_Support_Tickets.txt
Text snippet: Ticket_ID,Timestamp,Customer_ID,Subject,Message TKT-9001,2025-10-24 03:16:00,Cust_554,URGENT: 500 Error,I cannot log in ...
Distance: 0.6053494564476257
------------------------------------------------------------
ID: Slack_Channel_Ops.txt_chunk_2
Source: Slack_Channel_Ops.txt
Text snippet: Cha

Final Table Verification

In [ ]:
print("\n=== Oracle Vector Table Summary ===\n")

# --- 1. Table existence ---
cursor.execute("""
SELECT table_name
FROM user_tables
WHERE table_name IN ('CONTEXT_LIBRARY', 'KNOWLEDGE_STORE')
""")
print("Tables present:", [t[0] for t in cursor.fetchall()])


# --- 2. Schema for CONTEXT_LIBRARY ---
print("\n--- CONTEXT_LIBRARY Schema ---")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'CONTEXT_LIBRARY'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)


# --- 3. Schema for KNOWLEDGE_STORE ---
print("\n--- KNOWLEDGE_STORE Schema ---")
cursor.execute("""
SELECT column_name, data_type, data_length
FROM user_tab_columns
WHERE table_name = 'KNOWLEDGE_STORE'
ORDER BY column_id
""")
for row in cursor.fetchall():
    print(row)


# --- 4. Row counts ---
cursor.execute("SELECT COUNT(*) FROM context_library")
context_count = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM knowledge_store")
knowledge_count = cursor.fetchone()[0]

print(f"\nRows in CONTEXT_LIBRARY: {context_count}")
print(f"Rows in KNOWLEDGE_STORE: {knowledge_count}")


# --- 5. Sample rows (non‑vector fields only) ---
print("\n--- Sample from CONTEXT_LIBRARY ---")
cursor.execute("""
SELECT id, SUBSTR(description, 1, 120)
FROM context_library
FETCH FIRST 3 ROWS ONLY
""")
for row in cursor.fetchall():
    print(row)


print("\n--- Sample from KNOWLEDGE_STORE ---")
cursor.execute("""
SELECT id, source, SUBSTR(text, 1, 120)
FROM knowledge_store
FETCH FIRST 3 ROWS ONLY
""")
for row in cursor.fetchall():
    print(row)

print("\n=== Summary Complete ===")



=== Oracle Vector Table Summary ===

Tables present: ['CONTEXT_LIBRARY', 'KNOWLEDGE_STORE']

--- CONTEXT_LIBRARY Schema ---
('ID', 'VARCHAR2', 200)
('DESCRIPTION', 'CLOB', 4000)
('BLUEPRINT_JSON', 'CLOB', 4000)
('EMBEDDING', 'VECTOR', 8200)

--- KNOWLEDGE_STORE Schema ---
('ID', 'VARCHAR2', 200)
('SOURCE', 'VARCHAR2', 200)
('TEXT', 'CLOB', 4000)
('EMBEDDING', 'VECTOR', 8200)

Rows in CONTEXT_LIBRARY: 3
Rows in KNOWLEDGE_STORE: 7

--- Sample from CONTEXT_LIBRARY ---
('blueprint_public_pr', <oracledb.lob.LOB object at 0x7f9165e1b410>)
('blueprint_technical_rca', <oracledb.lob.LOB object at 0x7f9165e1b290>)
('blueprint_legal_compliance', <oracledb.lob.LOB object at 0x7f9165e196a0>)

--- Sample from KNOWLEDGE_STORE ---
('Email_Phishing_Report.txt_chunk_0', 'Email_Phishing_Report.txt', <oracledb.lob.LOB object at 0x7f9165e1b290>)
('Patch_Release_Notes.txt_chunk_1', 'Patch_Release_Notes.txt', <oracledb.lob.LOB object at 0x7f9165e1aea0>)
('Slack_Channel_Ops.txt_chunk_2', 'Slack_Channel_Ops.t